In [7]:
!pip install gradio
!pip install ultralytics
!pip install efficientnet-pytorch

import gradio as gr
from flask import Flask
from PIL import Image
import torch
import torchvision.transforms as transforms
import numpy as np
from ultralytics import YOLO
from io import BytesIO
import cv2

# Load Models
classification_model = torch.load('/content/EfficientNet_Without_Dropout.pth', map_location=torch.device('cpu'))
classification_model.eval()

yolo_detection_model = torch.hub.load('ultralytics/yolov5', 'custom', path='/content/YoloV5_best.pt', force_reload=True)
yolo_segmentation_model = YOLO('/content/YoloV8_best.pt')

# Define Preprocessing Transformations
classification_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Flood Classification Labels
flood_labels = {1: "Flood", 0: "No Flood"}

def classify_image(image):
    img_tensor = classification_transform(image).unsqueeze(0)
    with torch.no_grad():
        output = classification_model(img_tensor)
        prob_flood = torch.sigmoid(output).item()
        label = 1 if prob_flood > 0.5 else 0
        confidence = prob_flood if label == 1 else 1 - prob_flood

    return image, f"Label: {flood_labels[label]}, Confidence: {confidence * 100:.2f}%"

def detect_objects(image):
    results = yolo_detection_model(image)
    results.render()
    annotated_image = Image.fromarray(results.ims[0])
    detections = [
        f"{yolo_detection_model.names[int(cls_id)]}: {conf:.2f}"
        for *_, conf, cls_id in results.xyxy[0].cpu().numpy()
    ]
    return annotated_image, "\n".join(detections)

def segment_image(image):
    results = yolo_segmentation_model(image)
    masks = results[0].masks.data.cpu().numpy() if results[0].masks else None
    if masks is None:
        return image, "No segmentation results found."

    overlay_image = np.array(image)
    for mask in masks:
        mask = (cv2.resize(mask, (overlay_image.shape[1], overlay_image.shape[0])) * 255).astype(np.uint8)
        overlay_image[mask > 0] = (0, 255, 0)  # Apply green overlay

    overlay_img_pil = Image.fromarray(overlay_image)
    class_names = [results[0].names[int(cls_id)] for cls_id in results[0].boxes.cls.cpu().numpy().astype(int)]
    return overlay_img_pil, ", ".join(set(class_names))

def process_image(image, model_type):
    if model_type == "Classification":
        return classify_image(image)
    elif model_type == "Object Detection":
        return detect_objects(image)
    elif model_type == "Segmentation":
        return segment_image(image)

# Define Gradio Interface
model_choices = ["Classification", "Object Detection", "Segmentation"]

interface = gr.Interface(
    fn=process_image,
    inputs=[
        gr.Image(type="pil", label="Upload Image"),
        gr.Dropdown(choices=model_choices, label="Select Model")
    ],
    outputs=[
        gr.Image(type="pil", label="Processed Image"),
        gr.Text(label="Results")
    ],
    title="Smart Flood Detection System",
    description="Upload an image and select a model type to process it using classification, object detection, or segmentation."
)

# Launch Interface
if __name__ == "__main__":
    interface.launch(share=True)


  Preparing metadata (setup.py) ... done
  Created wheel for efficientnet-pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16424 sha256=48efc79f54bc34511070c032063b9f6ed96696d5de928dc9d831d1b86100059f
  Stored in directory: /root/.cache/pip/wheels/03/3f/e9/911b1bc46869644912bda90a56bcf7b960f20b5187feea3baf
Successfully built efficientnet-pytorch


/usr/local/lib/python3.10/dist-packages/torch/hub.py:330: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(
Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /root/.cache/torch/hub/master.zip
YOLOv5 🚀 2024-12-14 Python-3.10.12 torch-2.5.1+cu121 CPU

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fb4b858b3b238e0c68.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
